In [1]:
# ============================================================
# STEP 1 — IMPORT LIBRARIES
# ============================================================

import ee
import pandas as pd
import numpy as np
import os

C:\Users\Projit\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.5) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
# ============================================================
# STEP 2 — INITIALIZE GOOGLE EARTH ENGINE
# ============================================================


PROJECT_ID = "sundarbans-climate-analysis"

ee.Initialize(project=PROJECT_ID)


In [3]:
print(
    "Google Earth Engine initialized successfully!"
)

Google Earth Engine initialized successfully!


In [4]:
# ============================================================
# STEP 3 — LOAD BANGLADESH DISTRICT BOUNDARIES
# ============================================================

districts = ee.FeatureCollection(
    "FAO/GAUL/2015/level2"
)

C:\Users\Projit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level2! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level2

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level2

  warnings.warn(warning, category=DeprecationWarning)


In [5]:
# ============================================================
# STEP 4 — SELECT BANGLADESH
# ============================================================

bangladesh = districts.filter(
    ee.Filter.eq(
        "ADM0_NAME",
        "Bangladesh"
    )
)

In [6]:
# ============================================================
# STEP 5 — SELECT SATKHIRA, KHULNA & BAGERHAT
# ============================================================

study_districts = bangladesh.filter(
    ee.Filter.inList(
        "ADM2_NAME",
        [
            "Satkhira",
            "Khulna",
            "Bagerhat"
        ]
    )
)

In [7]:
# ============================================================
# STEP 6 — CREATE COMBINED STUDY AREA
# ============================================================

study_area = study_districts.geometry()


# Check selected districts
print(
    "Selected districts:"
)

print(
    study_districts
    .aggregate_array("ADM2_NAME")
    .getInfo()
)


Selected districts:


['Bagerhat', 'Khulna', 'Satkhira']


In [8]:
# ============================================================
# STEP 7 — LOAD MODIS MONTHLY NDVI DATA
# ============================================================

ndvi_collection = (
    ee.ImageCollection(
        "MODIS/061/MOD13A3"
    )
    .filterDate(
        "2000-01-01",
        "2026-01-01"
    )
    .select("NDVI")
)

In [9]:
# Check number of images
print(
    "Number of MODIS NDVI images:",
    ndvi_collection.size().getInfo()
)

Number of MODIS NDVI images: 311


In [10]:
# ============================================================
# STEP 8 — CREATE MONTH SEQUENCE
# ============================================================
# 26 years × 12 months = 312 months

months = ee.List.sequence(
    0,
    311
)

In [11]:
# ============================================================
# STEP 9 — CORRECTED FUNCTION
# ============================================================

def calculate_monthly_ndvi(n):

    n = ee.Number(n)

    # Month start
    start_date = (
        ee.Date("2000-01-01")
        .advance(n, "month")
    )

    # Month end
    end_date = (
        start_date.advance(1, "month")
    )

    # Collection for this month
    monthly_collection = (
        ndvi_collection
        .filterDate(
            start_date,
            end_date
        )
    )

    # Check whether an image exists
    image_count = monthly_collection.size()

    has_image = image_count.gt(0)

    # Get first image
    first_image = monthly_collection.first()

    # --------------------------------------------------------
    # IMPORTANT:
    # If no image exists, use a dummy image temporarily.
    # This prevents Image.multiply() from receiving null.
    # --------------------------------------------------------

    dummy_image = ee.Image.constant(0).rename("NDVI")

    image = ee.Image(
        ee.Algorithms.If(
            has_image,
            first_image,
            dummy_image
        )
    )

    # MODIS NDVI scale factor
    ndvi = image.multiply(0.0001)

    # Spatial mean
    result = ndvi.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=study_area,
        scale=1000,
        maxPixels=1e9,
        bestEffort=True,
        tileScale=4
    )

    # If image exists → actual NDVI
    # If image does not exist → null

    mean_ndvi = ee.Algorithms.If(
        has_image,
        result.get("NDVI"),
        None
    )

    # Return Feature
    return ee.Feature(
        None,
        {
            "Year":
                start_date.get("year"),

            "Month_Number":
                start_date.get("month"),

            "Month":
                start_date.format("MMMM"),

            "Date":
                start_date.format("YYYY-MM-dd"),

            "NDVI_Image_Count":
                image_count,

            "Mean_NDVI":
                mean_ndvi
        }
    )

In [12]:
# ============================================================
# STEP 10 — CREATE MONTHLY NDVI FEATURES
# ============================================================

months = ee.List.sequence(
    0,
    311
)

ndvi_features = ee.FeatureCollection(
    months.map(
        calculate_monthly_ndvi
    )
)

print(
    "Monthly NDVI FeatureCollection created successfully."
)

Monthly NDVI FeatureCollection created successfully.


In [13]:
# ============================================================
# STEP 11 — GET DATA
# ============================================================

ndvi_data = ndvi_features.getInfo()

print("Data successfully retrieved from Earth Engine!")

Data successfully retrieved from Earth Engine!


In [14]:
# ============================================================
# STEP 12 — CONVERT TO DATAFRAME
# ============================================================

ndvi_df = pd.DataFrame(
    [
        feature["properties"]
        for feature in ndvi_data["features"]
    ]
)

print(ndvi_df.head(10))

         Date  Mean_NDVI      Month  Month_Number  NDVI_Image_Count  Year
0  2000-01-01        NaN    January             1                 0  2000
1  2000-02-01   0.473782   February             2                 1  2000
2  2000-03-01   0.472318      March             3                 1  2000
3  2000-04-01   0.466707      April             4                 1  2000
4  2000-05-01   0.445410        May             5                 1  2000
5  2000-06-01   0.455811       June             6                 1  2000
6  2000-07-01   0.451224       July             7                 1  2000
7  2000-08-01   0.487742     August             8                 1  2000
8  2000-09-01   0.590108  September             9                 1  2000
9  2000-10-01   0.622527    October            10                 1  2000


In [15]:
# ============================================================
# STEP 13 — DATA TYPES
# ============================================================

ndvi_df["Year"] = pd.to_numeric(
    ndvi_df["Year"],
    errors="coerce"
)

ndvi_df["Month_Number"] = pd.to_numeric(
    ndvi_df["Month_Number"],
    errors="coerce"
)

ndvi_df["NDVI_Image_Count"] = pd.to_numeric(
    ndvi_df["NDVI_Image_Count"],
    errors="coerce"
)

ndvi_df["Mean_NDVI"] = pd.to_numeric(
    ndvi_df["Mean_NDVI"],
    errors="coerce"
)

In [16]:
# ============================================================
# STEP 14 — SORT
# ============================================================

ndvi_df = ndvi_df.sort_values(
    [
        "Year",
        "Month_Number"
    ]
).reset_index(
    drop=True
)

print(ndvi_df.head(12))

          Date  Mean_NDVI      Month  Month_Number  NDVI_Image_Count  Year
0   2000-01-01        NaN    January             1                 0  2000
1   2000-02-01   0.473782   February             2                 1  2000
2   2000-03-01   0.472318      March             3                 1  2000
3   2000-04-01   0.466707      April             4                 1  2000
4   2000-05-01   0.445410        May             5                 1  2000
5   2000-06-01   0.455811       June             6                 1  2000
6   2000-07-01   0.451224       July             7                 1  2000
7   2000-08-01   0.487742     August             8                 1  2000
8   2000-09-01   0.590108  September             9                 1  2000
9   2000-10-01   0.622527    October            10                 1  2000
10  2000-11-01   0.607298   November            11                 1  2000
11  2000-12-01   0.528703   December            12                 1  2000


In [17]:
# ============================================================
# STEP 15 — CHECK ROWS
# ============================================================

print(
    "Total records:",
    len(ndvi_df)
)

Total records: 312


In [18]:
# ============================================================
# STEP 16 — CHECK MISSING VALUES
# ============================================================

print(
    "Missing NDVI values:",
    ndvi_df["Mean_NDVI"].isna().sum()
)

Missing NDVI values: 1


In [19]:
# ============================================================
# STEP 17 — FIND MISSING MONTHS
# ============================================================

missing_months = ndvi_df[
    ndvi_df["Mean_NDVI"].isna()
]

print(
    missing_months
)

         Date  Mean_NDVI    Month  Month_Number  NDVI_Image_Count  Year
0  2000-01-01        NaN  January             1                 0  2000


In [20]:
# ============================================================
# STEP 18 — NDVI RANGE
# ============================================================

print(
    "Minimum NDVI:",
    ndvi_df["Mean_NDVI"].min()
)

print(
    "Maximum NDVI:",
    ndvi_df["Mean_NDVI"].max()
)

Minimum NDVI: 0.3720106547274133
Maximum NDVI: 0.6559499621715412


In [21]:
# ============================================================
# STEP 19 — ROUND NDVI
# ============================================================

ndvi_df["Mean_NDVI"] = ndvi_df[
    "Mean_NDVI"
].round(3)

In [22]:
# ============================================================
# STEP 20 — EXPORT TO EXCEL
# ============================================================

output_file = (
    "Sundarbans_Monthly_Vegetation_NDVI_2000_2025.xlsx"
)

ndvi_df.to_excel(
    output_file,
    index=False
)

print(
    "Excel file created successfully!"
)

print(
    "File:",
    output_file
)

Excel file created successfully!
File: Sundarbans_Monthly_Vegetation_NDVI_2000_2025.xlsx


In [23]:
print(
    ndvi_df.head(15)
)

          Date  Mean_NDVI      Month  Month_Number  NDVI_Image_Count  Year
0   2000-01-01        NaN    January             1                 0  2000
1   2000-02-01      0.474   February             2                 1  2000
2   2000-03-01      0.472      March             3                 1  2000
3   2000-04-01      0.467      April             4                 1  2000
4   2000-05-01      0.445        May             5                 1  2000
5   2000-06-01      0.456       June             6                 1  2000
6   2000-07-01      0.451       July             7                 1  2000
7   2000-08-01      0.488     August             8                 1  2000
8   2000-09-01      0.590  September             9                 1  2000
9   2000-10-01      0.623    October            10                 1  2000
10  2000-11-01      0.607   November            11                 1  2000
11  2000-12-01      0.529   December            12                 1  2000
12  2001-01-01      0.463

In [24]:
## EDA vegetatation

In [25]:
input_file = "Sundarbans_Monthly_Vegetation_NDVI_2000_2025.xlsx"


In [26]:
os.makedirs("EDA", exist_ok=True)


In [27]:
output_file = os.path.join(
    "EDA",
    "Vegetation_NDVI_EDA_2_1_Descriptive_Statistics.xlsx"
)



In [28]:
## LOAD DATA

In [29]:
df = pd.read_excel(input_file)

In [30]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

In [31]:
df["Mean_NDVI"] = pd.to_numeric(
    df["Mean_NDVI"], errors="coerce"
)

In [32]:
df["NDVI_Image_Count"] = pd.to_numeric(
    df["NDVI_Image_Count"], errors="coerce"
)

In [33]:
ndvi = df["Mean_NDVI"].dropna()


In [34]:
## DATASET OVERVIEW

In [35]:
overview = pd.DataFrame({
    "Metric": [
        "Total Observations",
        "Total Variables",
        "Start Date",
        "End Date",
        "Missing NDVI Values",
        "Duplicate Records",
        "Zero Image Count Records",
        "Minimum NDVI",
        "Maximum NDVI",
        "Mean NDVI"
    ],
    "Value": [
        len(df),
        len(df.columns),
        df["Date"].min(),
        df["Date"].max(),
        df["Mean_NDVI"].isna().sum(),
        df.duplicated().sum(),
        (df["NDVI_Image_Count"] == 0).sum(),
        ndvi.min(),
        ndvi.max(),
        ndvi.mean()
    ]
})


In [36]:
## NDVI DESCRIPTIVE STATISTICS

In [37]:
descriptive_stats = ndvi.describe().to_frame(
    name="NDVI"
)

descriptive_stats.loc["Range"] = (
    ndvi.max() - ndvi.min()
)

descriptive_stats.loc["IQR"] = (
    ndvi.quantile(0.75) -
    ndvi.quantile(0.25)
)

descriptive_stats.loc["Variance"] = ndvi.var()

descriptive_stats.loc["Standard Deviation"] = ndvi.std()

In [38]:
## DISTRIBUTION SHAPE

In [39]:
distribution = pd.DataFrame({
    "Metric": [
        "Mean",
        "Median",
        "Skewness",
        "Kurtosis"
    ],
    "Value": [
        ndvi.mean(),
        ndvi.median(),
        ndvi.skew(),
        ndvi.kurt()
    ]
})

In [40]:
## NDVI RANGE VALIDATION

In [41]:
df["NDVI_Range_Status"] = np.select(
    [
        df["Mean_NDVI"].isna(),
        (df["Mean_NDVI"] < -1) | (df["Mean_NDVI"] > 1)
    ],
    [
        "Missing NDVI",
        "Outside Theoretical Range"
    ],
    default="Within Theoretical Range"
)

In [42]:
range_issues = df[
    df["NDVI_Range_Status"] == "Outside Theoretical Range"
].copy()

In [43]:
range_summary = pd.DataFrame({
    "Metric": [
        "Theoretical Minimum NDVI",
        "Theoretical Maximum NDVI",
        "Values Below -1",
        "Values Above +1",
        "Total Values Outside Range"
    ],
    "Value": [
        -1,
        1,
        (df["Mean_NDVI"] < -1).sum(),
        (df["Mean_NDVI"] > 1).sum(),
        len(range_issues)
    ]
})


In [44]:
## IQR-BASED POTENTIAL OUTLIER DETECTION

In [45]:
Q1 = ndvi.quantile(0.25)
Q3 = ndvi.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

df["Outlier_Status"] = np.select(
    [
        df["Mean_NDVI"].isna(),
        (df["Mean_NDVI"] < lower_bound) |
        (df["Mean_NDVI"] > upper_bound)
    ],
    [
        "Missing NDVI",
        "Potential Outlier"
    ],
    default="Normal"
)

outliers = df[
    df["Outlier_Status"] == "Potential Outlier"
].copy()

outlier_summary = pd.DataFrame({
    "Metric": [
        "Q1",
        "Q3",
        "IQR",
        "Lower Bound",
        "Upper Bound",
        "Total Potential Outliers"
    ],
    "Value": [
        Q1,
        Q3,
        IQR,
        lower_bound,
        upper_bound,
        len(outliers)
    ]
})

In [46]:
## MONTHLY SUMMARY

In [47]:
monthly_summary = (
    df.groupby(["Month_Number", "Month"])["Mean_NDVI"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median",
        Standard_Deviation="std",
        Minimum="min",
        Maximum="max"
    )
    .reset_index()
    .sort_values("Month_Number")
)

In [48]:
## ANNUAL SUMMARY

In [49]:
annual_summary = (
    df.groupby("Year")["Mean_NDVI"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median",
        Standard_Deviation="std",
        Minimum="min",
        Maximum="max"
    )
    .reset_index()
)

In [50]:
## IMAGE AVAILABILITY SUMMARY

In [51]:
image_summary = pd.DataFrame({
    "Metric": [
        "Total Available Images",
        "Average Images Per Month",
        "Minimum Images Per Month",
        "Maximum Images Per Month",
        "Months With Zero Images",
        "Months With One Or More Images"
    ],
    "Value": [
        df["NDVI_Image_Count"].sum(),
        df["NDVI_Image_Count"].mean(),
        df["NDVI_Image_Count"].min(),
        df["NDVI_Image_Count"].max(),
        (df["NDVI_Image_Count"] == 0).sum(),
        (df["NDVI_Image_Count"] > 0).sum()
    ]
})


In [52]:
## EXPORT RESULTS TO EXCEL

In [53]:
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    overview.to_excel(
        writer, sheet_name="Dataset_Overview", index=False
    )

    descriptive_stats.to_excel(
        writer, sheet_name="Descriptive_Statistics"
    )

    distribution.to_excel(
        writer, sheet_name="Distribution_Shape", index=False
    )

    range_summary.to_excel(
        writer, sheet_name="NDVI_Range_Check", index=False
    )

    range_issues.to_excel(
        writer, sheet_name="Range_Issues", index=False
    )

    outlier_summary.to_excel(
        writer, sheet_name="Outlier_Summary", index=False
    )

    outliers.to_excel(
        writer, sheet_name="Potential_Outliers", index=False
    )

    monthly_summary.to_excel(
        writer, sheet_name="Monthly_Summary", index=False
    )

    annual_summary.to_excel(
        writer, sheet_name="Annual_Summary", index=False
    )

    image_summary.to_excel(
        writer, sheet_name="Image_Availability", index=False
    )

In [54]:
print("Vegetation NDVI EDA 2.1 completed successfully!")
print("Total observations:", len(df))
print("Valid NDVI observations:", ndvi.count())
print("Missing NDVI observations:", df["Mean_NDVI"].isna().sum())
print("Potential outliers:", len(outliers))
print("Report saved at:", os.path.abspath(output_file))

Vegetation NDVI EDA 2.1 completed successfully!
Total observations: 312
Valid NDVI observations: 311
Missing NDVI observations: 1
Potential outliers: 0
Report saved at: C:\projects_data\sundarbans_climate_analysis_gee\EDA\Vegetation_NDVI_EDA_2_1_Descriptive_Statistics.xlsx
